# 06 — Output Validation and Repair — Practical Examples

**Covers**: Three validation layers, self-repair loop, fallback chain, field sanitization, metrics tracking

In [ ]:
import os, json, re
from openai import OpenAI
from pydantic import BaseModel, Field, field_validator, model_validator, ValidationError
from typing import Optional, Literal
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o-mini'
print('✅ Setup complete')

## 1. Three-Layer Validation Stack

In [ ]:
class InvoiceLineItem(BaseModel):
    description: str
    quantity: int = Field(ge=1)
    unit_price: float = Field(ge=0)
    line_total: float

    @model_validator(mode='after')
    def validate_line_total(self):
        """Layer 3: Business logic — line_total must match qty * unit_price."""
        expected = round(self.quantity * self.unit_price, 2)
        actual   = round(self.line_total, 2)
        if abs(expected - actual) > 0.02:
            # Auto-correct instead of raising — LLM math isn't perfect
            self.line_total = expected
        return self

class Invoice(BaseModel):
    vendor: str
    invoice_id: str
    items: list[InvoiceLineItem]
    subtotal: float
    tax_rate: float = Field(ge=0, le=1)   # 0 to 100%
    total: float

    @model_validator(mode='after')
    def validate_totals(self):
        """Layer 3: subtotal and total must be consistent."""
        computed_subtotal = round(sum(i.line_total for i in self.items), 2)
        if abs(computed_subtotal - round(self.subtotal, 2)) > 0.05:
            self.subtotal = computed_subtotal
        expected_total = round(self.subtotal * (1 + self.tax_rate), 2)
        if abs(expected_total - round(self.total, 2)) > 0.05:
            self.total = expected_total
        return self

invoice_text = """
Invoice #INV-2024-1122 from Acme Supplies:
- 3x Office Chairs at $89.99 each = $269.97
- 2x Desks at $249.00 each = $498.00
Subtotal: $767.97, Tax: 8%, Total: $829.41
"""

r = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[{"role": "user", "content": f"Extract invoice:\n{invoice_text}"}],
    response_format=Invoice
)
inv = r.choices[0].message.parsed
print(f"Vendor:   {inv.vendor}")
print(f"Items:    {len(inv.items)}")
for item in inv.items:
    print(f"  {item.description:<25} qty={item.quantity} × ${item.unit_price:.2f} = ${item.line_total:.2f}")
print(f"Subtotal: ${inv.subtotal:.2f}")
print(f"Tax:      {inv.tax_rate:.0%}")
print(f"Total:    ${inv.total:.2f}")

## 2. Detection and Error Classification

In [ ]:
from enum import Enum

class ExtractionError(Enum):
    REFUSAL     = "refusal"
    TRUNCATION  = "truncation"
    VALIDATION  = "validation_error"
    API_ERROR   = "api_error"
    EMPTY       = "empty_result"
    SUCCESS     = "success"

def classified_extract(text: str, schema: type[BaseModel]) -> tuple[BaseModel | None, ExtractionError, str]:
    """Extract and classify any failure."""
    try:
        response = client.beta.chat.completions.parse(
            model=MODEL,
            messages=[{"role": "user", "content": text}],
            response_format=schema
        )
        msg = response.choices[0].message
        
        if msg.refusal:
            return None, ExtractionError.REFUSAL, msg.refusal
        if response.choices[0].finish_reason == "length":
            return None, ExtractionError.TRUNCATION, "max_tokens reached"
        
        result = msg.parsed
        if result is None:
            return None, ExtractionError.EMPTY, "Parsed object is None"
        return result, ExtractionError.SUCCESS, ""
    
    except ValidationError as e:
        return None, ExtractionError.VALIDATION, e.json()
    except Exception as e:
        return None, ExtractionError.API_ERROR, f"{type(e).__name__}: {e}"

class PersonProfile(BaseModel):
    name: str
    age: int = Field(ge=0, le=130)
    email: str
    role: Literal["engineer", "manager", "analyst", "designer", "other"]

test_inputs = [
    "Alice Chen, 34, alice@company.com, Senior Engineer",
    "Bob Smith, 45, bob@company.com, VP of Sales",  # role may map to 'manager' or 'other'
    ""   # empty - edge case
]

for text in test_inputs:
    result, err_type, err_msg = classified_extract(text or "No data", PersonProfile)
    status = "✅" if err_type == ExtractionError.SUCCESS else "❌"
    print(f"{status} Input: {text[:40]!r:<42} | {err_type.value}")
    if result:
        print(f"   → {result.name}, age={result.age}, role={result.role}")

## 3. Self-Repair Loop — LLM Fixes Its Own Mistakes

In [ ]:
def extract_with_repair(text: str, schema: type[BaseModel], max_repairs: int = 2) -> BaseModel | None:
    """Ask LLM to self-repair when validation fails."""
    messages = [
        {"role": "system", "content": "You are a precise data extractor. Follow the schema exactly."},
        {"role": "user",   "content": f"Extract data:\n{text}"}
    ]
    
    for attempt in range(max_repairs + 1):
        r = client.beta.chat.completions.parse(
            model=MODEL, messages=messages, response_format=schema
        )
        msg = r.choices[0].message
        
        if msg.refusal:
            print(f"  🚫 Refused: {msg.refusal[:80]}")
            return None
        
        result = msg.parsed
        if result is not None:
            label = "original" if attempt == 0 else f"repair #{attempt}"
            print(f"  ✅ Success ({label})")
            return result
        
        if attempt < max_repairs:
            raw_content = msg.content
            errs = "Schema mismatch — please re-check all required fields and types."
            messages.append({"role": "assistant", "content": raw_content or ""})
            messages.append({"role": "user",      "content": f"Your output had issues: {errs} Please redo it."})
            print(f"  ⚠️  Attempt {attempt+1} failed, sending repair prompt...")
    
    print(f"  ❌ Failed after {max_repairs+1} attempts")
    return None

class TechSpec(BaseModel):
    product: str
    cpu: str
    ram_gb: int = Field(ge=1)
    storage_gb: int = Field(ge=1)
    price_usd: float

spec_text = "MacBook Pro 14-inch: M3 Pro chip, 18GB unified memory, 512GB SSD. Price: $1,999"

print("Testing self-repair extraction:")
result = extract_with_repair(spec_text, TechSpec)
if result:
    print(f"  Product: {result.product}")
    print(f"  CPU: {result.cpu}, RAM: {result.ram_gb}GB, Storage: {result.storage_gb}GB")
    print(f"  Price: ${result.price_usd:,.2f}")

## 4. Fallback Chain — Graceful Degradation

In [ ]:
class EventSummary(BaseModel):
    name: str
    date: str
    location: str
    attendees: int

def extract_event_fallback(text: str) -> EventSummary | None:
    """Three-tier fallback: strict → JSON mode → prompt-only"""
    
    # Tier 1: Strict structured output
    try:
        r = client.beta.chat.completions.parse(
            model=MODEL,
            messages=[{"role": "user", "content": f"Extract event: {text}"}],
            response_format=EventSummary
        )
        if r.choices[0].message.parsed:
            print("  ✅ Tier 1 (strict) succeeded")
            return r.choices[0].message.parsed
    except Exception as e:
        print(f"  ⚠️  Tier 1 failed: {type(e).__name__}")
    
    # Tier 2: JSON mode + Pydantic coercion
    try:
        r = client.chat.completions.create(
            model=MODEL,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": "Extract event as JSON: name, date, location, attendees (int)."},
                {"role": "user",   "content": text}
            ]
        )
        raw = json.loads(r.choices[0].message.content)
        result = EventSummary(
            name=str(raw.get("name", "Unknown")),
            date=str(raw.get("date", "TBD")),
            location=str(raw.get("location", "Unknown")),
            attendees=int(raw.get("attendees", raw.get("attendee_count", 0)))
        )
        print("  ✅ Tier 2 (JSON mode) succeeded")
        return result
    except Exception as e:
        print(f"  ⚠️  Tier 2 failed: {type(e).__name__}")
    
    # Tier 3: Regex rescue from free text
    try:
        r = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": f"List event details line by line (Name:, Date:, Location:, Attendees:):\n{text}"}]
        )
        raw_text = r.choices[0].message.content
        lines = {line.split(':')[0].strip().lower(): ':'.join(line.split(':')[1:]).strip()
                 for line in raw_text.split('\n') if ':' in line}
        result = EventSummary(
            name=lines.get('name', 'Unknown'),
            date=lines.get('date', 'TBD'),
            location=lines.get('location', 'Unknown'),
            attendees=int(re.search(r'\d+', lines.get('attendees', '0')).group() or 0)
        )
        print("  ✅ Tier 3 (regex rescue) succeeded")
        return result
    except Exception as e:
        print(f"  ❌ Tier 3 failed: {type(e).__name__}")
    
    return None

print("Fallback chain test:")
event = extract_event_fallback(
    "Annual Tech Summit on March 15, 2025 at San Francisco Convention Center. Expected 2,500 attendees."
)
if event:
    print(f"Result: {event.name} | {event.date} | {event.location} | {event.attendees} attendees")

## 5. Pydantic Sanitization Validators in Action

In [ ]:
import re
from pydantic import field_validator

class CleanedContactCard(BaseModel):
    full_name: str
    email: str
    phone: Optional[str] = None
    website: Optional[str] = None
    company: str

    @field_validator('full_name')
    @classmethod
    def title_case(cls, v):
        return ' '.join(v.strip().split()).title()

    @field_validator('email')
    @classmethod
    def normalize_email(cls, v):
        return v.strip().lower()

    @field_validator('phone')
    @classmethod
    def clean_phone(cls, v):
        if not v: return None
        digits_only = re.sub(r'[^\d+]', '', v)
        return digits_only if len(digits_only) >= 7 else None

    @field_validator('website')
    @classmethod
    def normalize_url(cls, v):
        if not v: return None
        v = v.strip()
        if not v.startswith('http'):
            v = 'https://' + v
        return v.rstrip('/')

bizcard = """
MEET: SARAH O'BRIEN
Company: Innovate Corp
Email: Sarahobrien@Innovatecorp.COM
Phone: +1 (555) 234-5678 x42
Web: www.innovatecorp.com
"""

r = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[{"role": "user", "content": f"Extract contact info:\n{bizcard}"}],
    response_format=CleanedContactCard
)
card = r.choices[0].message.parsed
print(f"Name:    {card.full_name!r}    ← title-cased")
print(f"Email:   {card.email!r}  ← lowercased")
print(f"Phone:   {card.phone!r}    ← digits only")
print(f"Website: {card.website!r}")

## 6. Lightweight Metrics Tracker

In [ ]:
import time
from dataclasses import dataclass, field

@dataclass
class ExtractionMetrics:
    total: int = 0
    successes: int = 0
    failures: int = 0
    repairs: int = 0
    latencies: list[float] = field(default_factory=list)

    def record(self, success: bool, repaired: bool, latency_ms: float):
        self.total += 1
        self.successes += int(success)
        self.failures  += int(not success)
        self.repairs   += int(repaired)
        self.latencies.append(latency_ms)

    def report(self):
        avg = sum(self.latencies) / len(self.latencies) if self.latencies else 0
        print(f"\n{'─'*40}")
        print(f"  Extractions:   {self.total}")
        print(f"  Success rate:  {self.successes/self.total:.0%}" if self.total else "  No data")
        print(f"  Repaired:      {self.repairs}")
        print(f"  Avg latency:   {avg:.0f}ms")
        print(f"{'─'*40}")

metrics = ExtractionMetrics()

class SimpleProduct(BaseModel):
    name: str
    price: float

test_cases = [
    "iPhone 15 Pro, $999",
    "AirPods Pro Gen 2 at $249.99",
    "MacBook Air M2 — costs $1,099",
    "Apple Watch Series 9 — $399"
]

for text in test_cases:
    t_start = time.perf_counter()
    result, err_type, _ = classified_extract(text, SimpleProduct)
    latency = (time.perf_counter() - t_start) * 1000
    success = err_type == ExtractionError.SUCCESS
    metrics.record(success, False, latency)
    status = "✅" if success else "❌"
    name = result.name if result else "FAILED"
    print(f"{status} {name:<35} {latency:.0f}ms")

metrics.report()

## 📌 Summary

- **3 validation layers**: JSON syntax → Pydantic types → business logic (`@model_validator`)
- **Self-repair**: on failure, append error message to conversation and retry
- **Fallback chain**: strict → JSON mode → regex/prompt rescue
- **Field validators**: sanitize dirty LLM output (emails, phones, names)
- **Track metrics**: success rate, repair rate, latency — catch model drift early
- **Never crash**: return `None` + classified error in production code